In [1]:
from pathlib import Path
import pandas as pd

results_dir = Path("assets", "results")

if not results_dir.exists():
    raise FileNotFoundError(f"Results directory not found: {results_dir}")

id_columns = {
    "unique_id",
    "id",
    "series_id",
    "timestamp",
    "time",
    "date",
    "ds",
    "y",
    "target",
    "split",
    "fold",
    "horizon",
}

rows = []

for file in sorted(results_dir.glob("*/scores*.csv")):
    df = pd.read_csv(file)

    method_columns = [
        col
        for col in df.columns
        if col not in id_columns and pd.api.types.is_numeric_dtype(df[col])
    ]

    if not method_columns:
        continue

    row = {
        "file": file.name,
        "dataset": file.stem.replace("scores,", "")
        .replace("scores_", "")
        .replace("scores-", ""),
        "n_series": len(df),
    }

    row.update(df[method_columns].mean(numeric_only=True).to_dict())
    rows.append(row)

raw_summary = pd.DataFrame(rows)

method_columns = [
    col for col in raw_summary.columns if col not in {"file", "dataset", "n_series"}
]

summary = raw_summary.groupby(["file", "dataset"], as_index=False).agg(
    {"n_series": "max", **{col: "mean" for col in method_columns}}
)

best_two = summary[method_columns].apply(
    lambda row: row.dropna().nsmallest(2).index.tolist(), axis=1
)
best_two = best_two.apply(lambda x: x + [pd.NA] * (2 - len(x)))

summary[["best_model", "second_best_model"]] = pd.DataFrame(
    best_two.tolist(), index=summary.index
)

summary = (
    summary[
        ["dataset", "file", "n_series", *method_columns, "best_model", "second_best_model"]
    ]
    .sort_values("dataset")
    .reset_index(drop=True)
)

summary.round(2)

,dataset,file,n_series,MetaARIMA,AutoARIMA,SeasonalNaive,Chronos2,Moirai2,TimesFM,best_model,second_best_model
0,monash_m1_monthly,"scores,monash_m1_monthly.csv",617,0.88,0.92,1.12,0.87,0.97,0.90,Chronos2,MetaARIMA
1,monash_m3_monthly,"scores,monash_m3_monthly.csv",1428,0.71,0.74,1.00,0.69,0.74,0.72,Chronos2,MetaARIMA
2,monash_tourism_monthly,"scores,monash_tourism_monthly.csv",366,1.17,1.22,1.34,1.17,1.25,1.36,Chronos2,MetaARIMA
